# Probability Concepts & Bayes' Theorem (CFA Level 1)
## Foundations of Probability for Portfolio Management and Risk Analysis

---

### Why Probability Matters in Finance

Every investment decision is a bet on the future -- and the future is uncertain. Probability is the mathematical language we use to reason about that uncertainty.

When a portfolio manager says "there's a 20% chance of recession," or a credit analyst estimates "the probability of default is 2%," they are using probability concepts. Without this framework, investment decisions would be pure guesswork.

> **Key Concept:** Probability quantifies uncertainty. In finance, it is the foundation for risk assessment, portfolio construction, option pricing, and credit analysis. Every "what could go wrong?" question is fundamentally a probability question.

**What you will learn:**
1. Probability axioms, addition and multiplication rules
2. Conditional probability and independence
3. Bayes' theorem: derivation, implementation, and financial applications
4. Expected value, variance, covariance, and portfolio theory foundations
5. Counting methods for combinatorial finance problems

**Real-world motivation:** Imagine you are an analyst who has built a quantitative stock screen. The screen flags a stock as a "buy." But how confident should you be? The screen catches 80% of winners, but it also gives false positives on 15% of losers. Bayes' theorem tells you the true probability of success -- and the answer may surprise you.

**Prerequisites:** Basic algebra.

**References:**
- CFA Institute, *CFA Program Curriculum*, Quantitative Methods.
- DeFusco, R. et al., *Quantitative Investment Analysis*, CFA Institute, Wiley.

In [ ]:
%matplotlib inline
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── Reproducibility
SEED = 42
rng = np.random.default_rng(SEED)

# ── Tolerances
ATOL = 1e-10
RTOL = 1e-6

# ── Plot Style
PRIMARY   = 'steelblue'
SECONDARY = 'coral'
TERTIARY  = 'seagreen'
ACCENT    = 'gold'
plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12, 'axes.grid': True, 'grid.alpha': 0.3})

---
## 1. Probability Fundamentals

### What Is Probability?

At its core, probability assigns a number between 0 and 1 to events, measuring how likely they are to occur. A probability of 0 means "impossible" and 1 means "certain."

In finance, there are three main interpretations of probability:
- **Frequentist:** The long-run relative frequency of an event. "If we observe 10,000 similar bonds, about 200 will default" (2% default rate).
- **Subjective:** A personal degree of belief. "I believe there's a 30% chance the Fed raises rates." This is common in investment analysis.
- **A priori:** Based on logical analysis. "A fair coin has a 50% chance of heads."

### Sample Space and Events

- **Sample space** $\Omega$: the set of all possible outcomes (e.g., all possible stock returns tomorrow)
- **Event** $A \subseteq \Omega$: a subset of outcomes we care about (e.g., "the stock goes up")

### Axioms of Probability (Kolmogorov)

These three axioms are the foundation of all probability theory:

1. $0 \leq P(A) \leq 1$ for any event $A$
2. $P(\Omega) = 1$ (something must happen)
3. For mutually exclusive events $A_1, A_2, \ldots$: $P(A_1 \cup A_2 \cup \cdots) = \sum P(A_i)$

Everything else in probability theory is derived from these three axioms.

### Addition Rule: "Or" Probabilities

What is the probability that event $A$ **or** event $B$ (or both) occurs?

$$P(A \cup B) = P(A) + P(B) - P(A \cap B)$$

We subtract $P(A \cap B)$ to avoid double-counting the overlap.

If $A$ and $B$ are **mutually exclusive** (they can't both happen): $P(A \cup B) = P(A) + P(B)$.

**Worked Example:** In a credit portfolio, P(Bond A defaults) = 5%, P(Bond B defaults) = 8%, P(both default) = 2%.

$$P(\text{at least one defaults}) = 0.05 + 0.08 - 0.02 = 0.11 = 11\%$$

### Multiplication Rule: "And" Probabilities

$$P(A \cap B) = P(A|B) \cdot P(B) = P(B|A) \cdot P(A)$$

### Complement Rule: "Not" Probabilities

$$P(A') = 1 - P(A)$$

In the example above: $P(\text{neither defaults}) = 1 - 0.11 = 0.89 = 89\%$.

> **CFA Exam Tip:** The addition rule is one of the most commonly tested formulas. Always check whether events are mutually exclusive before simplifying. If they are, the joint probability $P(A \cap B) = 0$.

Let's implement these rules and verify with simulation.

In [ ]:
# ── Probability rules: Implementation and verification via simulation

def addition_rule(p_a, p_b, p_a_and_b):
    """P(A ∪ B) = P(A) + P(B) - P(A ∩ B)"""
    return p_a + p_b - p_a_and_b


def complement_rule(p_a):
    """P(A') = 1 - P(A)"""
    return 1 - p_a


# ── Example: Bond default probabilities
# P(bond A defaults) = 0.05, P(bond B defaults) = 0.08
# P(both default) = 0.02 (positive default correlation)
p_a, p_b, p_ab = 0.05, 0.08, 0.02

p_a_or_b = addition_rule(p_a, p_b, p_ab)
p_neither = 1 - p_a_or_b

print("Bond Default Probabilities:")
print(f"  P(A defaults)        = {p_a:.4f}")
print(f"  P(B defaults)        = {p_b:.4f}")
print(f"  P(both default)      = {p_ab:.4f}")
print(f"  P(at least one)      = {p_a_or_b:.4f}")
print(f"  P(neither defaults)  = {p_neither:.4f}")

# ── Verify via simulation
n_sim = 1_000_000
# Direct simulation from joint table:
# P(A∩B) = 0.02, P(A∩B') = 0.03, P(A'∩B) = 0.06, P(A'∩B') = 0.89
joint_probs = np.array([0.02, 0.03, 0.06, 0.89])
cum_probs = np.cumsum(joint_probs)
categories = np.digitize(rng.random(n_sim), cum_probs)

sim_p_a = np.mean((categories == 0) | (categories == 1))
sim_p_b = np.mean((categories == 0) | (categories == 2))
sim_p_ab = np.mean(categories == 0)
sim_p_a_or_b = np.mean(categories != 3)

print(f"\nSimulation verification (n={n_sim:,}):")
print(f"  P(A)          : theory={p_a:.4f}, sim={sim_p_a:.4f}")
print(f"  P(B)          : theory={p_b:.4f}, sim={sim_p_b:.4f}")
print(f"  P(A∩B)        : theory={p_ab:.4f}, sim={sim_p_ab:.4f}")
print(f"  P(A∪B)        : theory={p_a_or_b:.4f}, sim={sim_p_a_or_b:.4f}")

**Interpreting the output:** The simulation with 1 million trials closely matches the theoretical values, confirming our formulas work correctly. The slight differences are just random sampling noise -- they would disappear with even more simulations.

Notice the key financial insight: even though each bond has a relatively low default probability (5% and 8%), the probability that **at least one** defaults is 11%. This is higher than either individual probability because there are *two* things that can go wrong. In a portfolio of many bonds, diversification doesn't eliminate default risk -- it reduces it but never to zero.

---
## 2. Conditional Probability

### The Core Question: "What changes when I learn something new?"

Conditional probability answers the question: *"Given that I know B has happened, what is the probability of A?"*

This is fundamentally important in finance. For example:
- Given that we're in a recession, what is the probability this bond defaults?
- Given that a company's earnings exceeded estimates, what is the probability the stock goes up?
- Given that the Fed raised rates, what is the probability of another rate hike?

### The Formula

$$P(A|B) = \frac{P(A \cap B)}{P(B)}, \quad P(B) > 0$$

**Intuition:** We're "zooming in" on the world where $B$ has already happened. Out of all the scenarios where $B$ occurs, what fraction also has $A$?

**Worked Example:** From the joint probability table below:

|  | Recession (R) | Expansion (E) | Total |
|---|---|---|---|
| **Default (D)** | 0.06 | 0.02 | 0.08 |
| **No Default (D')** | 0.19 | 0.73 | 0.92 |
| **Total** | 0.25 | 0.75 | 1.00 |

$$P(D|R) = \frac{P(D \cap R)}{P(R)} = \frac{0.06}{0.25} = 0.24 = 24\%$$

$$P(D|E) = \frac{P(D \cap E)}{P(E)} = \frac{0.02}{0.75} = 0.027 = 2.7\%$$

The default probability jumps from 2.7% in an expansion to **24%** in a recession -- nearly 9 times higher! This is why the economic cycle matters so much for credit analysis.

### Independence

Events $A$ and $B$ are **independent** if and only if:
$$P(A|B) = P(A) \quad \Longleftrightarrow \quad P(A \cap B) = P(A) \cdot P(B)$$

**Intuition:** Learning that $B$ occurred gives you *no* new information about $A$.

In the example above: $P(D) \cdot P(R) = 0.08 \times 0.25 = 0.02$, but $P(D \cap R) = 0.06$. These are not equal, so default and recession are **not independent** -- they are positively associated.

> **Key Concept:** Independence means knowing one event tells you nothing about the other. In finance, most events are *not* independent -- that's what makes risk management so challenging. Defaults tend to cluster during recessions. Stock prices tend to fall together during crises.

> **Common Mistake:** Assuming independence when events are actually correlated. The 2008 financial crisis occurred partly because risk models assumed mortgage defaults were nearly independent -- they weren't.

Let's verify these calculations with code.

In [ ]:
def conditional_prob(p_a_and_b, p_b):
    """P(A|B) = P(A ∩ B) / P(B)"""
    assert p_b > 0, "P(B) must be positive"
    return p_a_and_b / p_b


def check_independence(p_a, p_b, p_a_and_b):
    """Check if A and B are independent: P(A∩B) = P(A)·P(B)?"""
    return np.isclose(p_a_and_b, p_a * p_b)


# ── Example: Default given economic state
joint_table = {
    'D_and_R': 0.06,  'D_and_E': 0.02,
    'ND_and_R': 0.19, 'ND_and_E': 0.73,
}

p_D = joint_table['D_and_R'] + joint_table['D_and_E']   # 0.08
p_R = joint_table['D_and_R'] + joint_table['ND_and_R']  # 0.25
p_E = joint_table['D_and_E'] + joint_table['ND_and_E']  # 0.75

# Conditional probabilities
p_D_given_R = conditional_prob(joint_table['D_and_R'], p_R)
p_D_given_E = conditional_prob(joint_table['D_and_E'], p_E)

print("Joint Probability Table: Default vs Economic State\n")
print(f"{'':>12} {'Recession':>12} {'Expansion':>12} {'Total':>8}")
print("-" * 46)
print(f"{'Default':>12} {joint_table['D_and_R']:>12.4f} {joint_table['D_and_E']:>12.4f} {p_D:>8.4f}")
print(f"{'No Default':>12} {joint_table['ND_and_R']:>12.4f} {joint_table['ND_and_E']:>12.4f} {1-p_D:>8.4f}")
print(f"{'Total':>12} {p_R:>12.4f} {p_E:>12.4f} {1.0:>8.4f}")

print(f"\nConditional Probabilities:")
print(f"  P(Default | Recession)  = {p_D_given_R:.4f}")
print(f"  P(Default | Expansion)  = {p_D_given_E:.4f}")
print(f"  P(Default) [unconditional] = {p_D:.4f}")
print(f"\n  Are Default and Recession independent? {check_independence(p_D, p_R, joint_table['D_and_R'])}")
print(f"  P(D)·P(R) = {p_D * p_R:.4f} vs P(D∩R) = {joint_table['D_and_R']:.4f}")

**Interpreting the output:**

The unconditional default probability is 8%. But conditional on a recession, it jumps to 24%. Conditional on an expansion, it drops to just 2.7%. This dramatic difference shows that the economic state provides very useful information about default risk.

The independence test confirms that $P(D) \times P(R) = 0.02 \neq P(D \cap R) = 0.06$. Default and recession are not independent -- they are positively associated (recessions cause more defaults).

---
## 3. Bayes' Theorem

### The Big Idea: Updating Beliefs with Evidence

Bayes' theorem is one of the most powerful ideas in all of statistics. It answers the question: *"Given that I just observed new evidence, how should I update my beliefs?"*

**The analogy:** Imagine you're a detective. You start with some initial suspects (your "prior" beliefs). Then new evidence arrives -- say, fingerprints at the scene. Bayes' theorem tells you exactly how to update your suspicion of each suspect based on the new evidence. The more consistent the evidence is with a particular suspect, the more your suspicion shifts toward them.

### Derivation

From the definition of conditional probability:
$$P(A|B) = \frac{P(A \cap B)}{P(B)} = \frac{P(B|A) \cdot P(A)}{P(B)}$$

Using the **law of total probability** for the denominator:
$$P(B) = \sum_{i=1}^{n} P(B|A_i) \cdot P(A_i)$$

where $\{A_1, \ldots, A_n\}$ partition the sample space.

### Bayes' Theorem (Full Form)

$$\boxed{P(A_i|B) = \frac{P(B|A_i) \cdot P(A_i)}{\sum_{j=1}^{n} P(B|A_j) \cdot P(A_j)}}$$

**Terminology:**
- $P(A_i)$: **Prior** probability (your belief before seeing the evidence)
- $P(B|A_i)$: **Likelihood** (how likely the evidence is under each hypothesis)
- $P(A_i|B)$: **Posterior** probability (your updated belief after seeing the evidence)
- $P(B)$: **Evidence** (total probability of seeing the data)

The formula can be remembered as:

$$\text{Posterior} = \frac{\text{Likelihood} \times \text{Prior}}{\text{Evidence}}$$

### The Classic Example: Adapted for Finance

A disease screening test is the classic way to build Bayes' intuition. Let's use it first, then apply the same logic to finance.

**Setup:** A test for a rare condition has:
- Prevalence: P(Disease) = 1% (the "base rate")
- Sensitivity: P(Test+ | Disease) = 95% (catches 95% of sick people)
- False positive rate: P(Test+ | No Disease) = 5%

**Question:** If you test positive, what is the probability you actually have the disease?

**Most people guess 90-95%.** The correct answer is shockingly low -- let's work through it step by step.

**Step 1:** Imagine 10,000 people taking the test.
- 100 have the disease (1% of 10,000)
- 9,900 don't have it

**Step 2:** Of the 100 with disease, 95 test positive (95% sensitivity).

**Step 3:** Of the 9,900 without disease, 495 test positive (5% false positive rate).

**Step 4:** Total positive tests: 95 + 495 = 590.

**Step 5:** P(Disease | Test+) = 95 / 590 = **16.1%**

> **Key Concept:** When the base rate is low, even a very accurate test produces mostly false positives. This same principle applies to stock screens, trading signals, and risk models. A "95% accurate" trading signal might still be wrong most of the time if profitable trades are rare.

Let's implement Bayes' theorem and visualize it as a probability tree.

In [ ]:
def bayes_theorem(priors, likelihoods):
    """Compute posterior probabilities using Bayes' theorem.
    
    Parameters
    ----------
    priors      : array-like -- P(A_i) for each hypothesis
    likelihoods : array-like -- P(B|A_i) for each hypothesis
    
    Returns
    -------
    posteriors  : ndarray -- P(A_i|B) for each hypothesis
    evidence    : float   -- P(B) = total probability of evidence
    """
    priors = np.asarray(priors, dtype=float)
    likelihoods = np.asarray(likelihoods, dtype=float)
    
    assert np.isclose(np.sum(priors), 1.0), "Priors must sum to 1"
    
    # Joint probabilities: P(B|A_i) * P(A_i)
    joint = likelihoods * priors
    
    # Evidence: P(B) = sum of joints
    evidence = np.sum(joint)
    
    # Posteriors: P(A_i|B) = joint_i / evidence
    posteriors = joint / evidence
    
    return posteriors, evidence


# ── Classic screening test example
priors = [0.01, 0.99]           # [P(Disease), P(No Disease)]
likelihoods = [0.95, 0.05]      # [P(+|Disease), P(+|No Disease)]

posteriors, p_positive = bayes_theorem(priors, likelihoods)

print("Classic Bayes Example: Screening Test\n")
print(f"  Prior P(Disease)       = {priors[0]:.4f}")
print(f"  P(Test+|Disease)       = {likelihoods[0]:.4f}")
print(f"  P(Test+|No Disease)    = {likelihoods[1]:.4f}")
print(f"\n  P(Test+) [evidence]    = {p_positive:.4f}")
print(f"  P(Disease|Test+)       = {posteriors[0]:.4f}")
print(f"  P(No Disease|Test+)    = {posteriors[1]:.4f}")
print(f"\n  Insight: Despite a positive test, there is only a {posteriors[0]:.1%} chance of disease!")
print(f"  The low base rate (1%) overwhelms the test's accuracy.")

**Why is this result so counterintuitive?** The key is the base rate. With only 1% prevalence, the "pool" of healthy people is 99 times larger than the pool of sick people. Even though the test is wrong on only 5% of healthy people, 5% of a very large number still produces more false positives than true positives.

This is directly relevant to finance: if only 30% of stocks outperform the benchmark, a stock screen that's "80% accurate" will still have a significant false positive rate. We'll see this exact application shortly.

The following probability tree makes the calculation visually clear.

In [ ]:
# ── Visualization: Bayes' theorem as a tree diagram
fig, ax = plt.subplots(figsize=(12, 7))
ax.set_xlim(-0.5, 3.5)
ax.set_ylim(-1.5, 1.5)
ax.axis('off')
ax.set_title("Bayes' Theorem: Probability Tree", fontsize=14, fontweight='bold')

# Tree structure
ax.annotate('', xy=(0.5, 0.7), xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color=PRIMARY, lw=2))
ax.annotate('', xy=(0.5, -0.7), xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color=SECONDARY, lw=2))

ax.text(0.15, 0.45, f'P(D) = {priors[0]}', fontsize=11, color=PRIMARY)
ax.text(0.05, -0.55, f"P(D') = {priors[1]}", fontsize=11, color=SECONDARY)

for y_start, p_pos, p_neg, color, label in [
    (0.7, likelihoods[0], 1-likelihoods[0], PRIMARY, 'D'),
    (-0.7, likelihoods[1], 1-likelihoods[1], SECONDARY, "D'")]:
    
    ax.annotate('', xy=(1.8, y_start + 0.3), xytext=(0.6, y_start),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.5))
    ax.annotate('', xy=(1.8, y_start - 0.3), xytext=(0.6, y_start),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.5, linestyle='--'))
    ax.text(0.9, y_start + 0.35, f'P(+|{label})={p_pos}', fontsize=10)
    ax.text(0.9, y_start - 0.2, f'P(-|{label})={p_neg}', fontsize=10)

joints = [
    (1.0, f'P(D∩+) = {priors[0]*likelihoods[0]:.4f}'),
    (0.4, f'P(D∩-) = {priors[0]*(1-likelihoods[0]):.4f}'),
    (-0.4, f"P(D'∩+) = {priors[1]*likelihoods[1]:.4f}"),
    (-1.0, f"P(D'∩-) = {priors[1]*(1-likelihoods[1]):.4f}"),
]
for y, text in joints:
    ax.text(2.0, y, text, fontsize=11, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

ax.text(0.5, -1.3, f"Posterior: P(D|+) = P(D∩+)/P(+) = {priors[0]*likelihoods[0]:.4f}/{p_positive:.4f} = {posteriors[0]:.4f}",
        fontsize=12, fontweight='bold', color='darkred',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='mistyrose', alpha=0.8))

plt.tight_layout()
plt.show()

**Reading the probability tree:**

Follow the branches from left to right:
1. The first branch splits by disease status (1% vs 99%).
2. The second branch splits by test result (positive/negative) conditional on disease status.
3. At the leaves, we get the joint probabilities by multiplying along the path.
4. The posterior is computed by dividing the joint probability of interest by the total probability of the evidence.

The key visual insight: the "Disease and Test+" leaf (0.0095) is **much smaller** than the "No Disease and Test+" leaf (0.0495). This is why most positive tests are false positives.

---
## 4. Bayes in Finance

Now let's apply Bayes' theorem to three practical financial scenarios. The mathematics is identical to the screening test -- only the context changes.

### Application 1: Stock Screening

You build a quantitative screen that flags stocks as "buy" candidates.
- **Base rate:** Only 30% of stocks actually outperform the benchmark
- **Sensitivity:** The screen correctly flags 80% of the stocks that will outperform
- **False positive rate:** The screen also flags 15% of the stocks that will underperform

**Question:** If your screen flags a stock, what is the probability it actually outperforms?

### Application 2: Updating Economic State Beliefs

Your prior belief: 20% chance of recession. Then a leading economic indicator turns negative.
- P(Negative indicator | Recession) = 70%
- P(Negative indicator | Expansion) = 10%

**Question:** After seeing the negative indicator, what is the updated recession probability?

### Application 3: Manager Skill Assessment

Only 10% of fund managers have true skill (generate alpha). A manager has outperformed 3 years in a row.
- P(Outperform 3 years | Skilled) = 75%
- P(Outperform 3 years | Unskilled) = 12.5% (just luck -- 0.5^3)

**Question:** After 3 years of outperformance, what is the probability the manager is truly skilled?

> **CFA Exam Tip:** Bayes' theorem problems on the exam always follow the same pattern: you're given priors and likelihoods, and asked to compute a posterior. Set up the formula methodically -- identify the prior, the likelihood, and compute the evidence.

Let's compute all three.

In [ ]:
# ── Application 1: Stock screening test
priors_stock = [0.30, 0.70]
likelihoods_stock = [0.80, 0.15]

posteriors_stock, p_screen_pos = bayes_theorem(priors_stock, likelihoods_stock)

print("Stock Screening Analysis:")
print(f"  Prior P(Outperform)          = {priors_stock[0]:.2f}")
print(f"  P(Screen+|Outperform)        = {likelihoods_stock[0]:.2f}")
print(f"  P(Screen+|Underperform)      = {likelihoods_stock[1]:.2f}")
print(f"  P(Screen+)                   = {p_screen_pos:.4f}")
print(f"  P(Outperform|Screen+)        = {posteriors_stock[0]:.4f}")
print(f"\n  The screen raises confidence from {priors_stock[0]:.0%} to {posteriors_stock[0]:.1%}")

# ── Application 2: Updating recession probability
priors_econ = [0.20, 0.80]
likelihoods_econ = [0.70, 0.10]

posteriors_econ, p_neg = bayes_theorem(priors_econ, likelihoods_econ)

print(f"\nEconomic State Update (after negative leading indicator):")
print(f"  Prior P(Recession)          = {priors_econ[0]:.2f}")
print(f"  Posterior P(Recession|Neg)   = {posteriors_econ[0]:.4f}")
print(f"  Recession probability jumped from {priors_econ[0]:.0%} to {posteriors_econ[0]:.1%}")

# ── Application 3: Manager skill
priors_mgr = [0.10, 0.90]
likelihoods_mgr = [0.75, 0.125]

posteriors_mgr, p_outperf = bayes_theorem(priors_mgr, likelihoods_mgr)

print(f"\nManager Skill Assessment (after 3 years outperformance):")
print(f"  Prior P(Skilled)             = {priors_mgr[0]:.2f}")
print(f"  Posterior P(Skilled|3yr out)  = {posteriors_mgr[0]:.4f}")
print(f"  Still only a {posteriors_mgr[0]:.1%} chance the manager is truly skilled!")

**Interpreting the three applications:**

1. **Stock screening:** The screen raises your confidence from 30% to about 70%. That's useful, but it still means nearly 1 in 3 flagged stocks will underperform. A naive investor might think an "80% accurate" screen means 80% of its picks will be winners -- Bayes' theorem shows the truth is more nuanced.

2. **Economic update:** A single negative indicator more than doubles the recession probability from 20% to about 64%. This shows how a highly diagnostic signal (one that's 7x more likely in a recession than an expansion) can dramatically shift beliefs.

3. **Manager skill:** Even after 3 years of outperformance, there's still only about a 40% chance the manager is truly skilled! This is because the prior (only 10% of managers have skill) is so low. The lesson: **track records are weaker evidence of skill than most investors assume.**

> **Common Mistake:** Ignoring the base rate. A 3-year track record of outperformance might feel convincing, but when 90% of managers are unskilled, even luck can produce impressive short-term results.

### Sequential Updating: How Many Years of Outperformance Does It Take?

What if the manager keeps beating the benchmark year after year? Let's track how the posterior evolves over time.

In [ ]:
# ── Visualization: Sequential Bayesian updating
p_skilled = 0.10  # prior
p_beat_skilled = 0.65    # P(beat | skilled)
p_beat_unskilled = 0.50  # P(beat | unskilled) = coin flip

years = 15
posterior_history = [p_skilled]

for year in range(years):
    posteriors, _ = bayes_theorem(
        [p_skilled, 1 - p_skilled],
        [p_beat_skilled, p_beat_unskilled]
    )
    p_skilled = posteriors[0]
    posterior_history.append(p_skilled)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(years + 1), posterior_history, 'o-', color=PRIMARY, linewidth=2, markersize=8)
ax.axhline(0.5, color=SECONDARY, linestyle='--', alpha=0.7, label='50% threshold')
ax.set_xlabel('Years of Consecutive Outperformance')
ax.set_ylabel('P(Manager is Skilled)')
ax.set_title('Sequential Bayesian Update: Manager Skill Assessment')
ax.set_ylim(0, 1)
ax.legend()

for i in [0, 5, 10, years]:
    ax.annotate(f'{posterior_history[i]:.1%}', (i, posterior_history[i]),
                textcoords='offset points', xytext=(0, 15), ha='center', fontsize=10)

plt.tight_layout()
plt.show()

**Interpreting the chart:**

Starting from just 10% confidence in the manager's skill, each year of outperformance nudges the posterior upward. It takes about **6-7 years** of consecutive outperformance just to reach 50% confidence! And even after 15 years of beating the benchmark every single year, we're still not at 100% -- there's always some residual probability of prolonged luck.

This has profound implications for the investment industry:
- Short track records are nearly meaningless for distinguishing skill from luck
- The prior (base rate of skilled managers) matters enormously
- This explains why even Nobel laureate-run funds (like LTCM) can blow up -- past performance truly does not guarantee future results

> **CFA Exam Tip:** Bayes' theorem on the CFA exam is always about updating probabilities with new information. The pattern: Prior + Evidence = Posterior. Practice setting up the formula with the correct prior and likelihood.

---
## 5. Expected Value & Variance

### Expected Value: The Probability-Weighted Average

The expected value of a random variable is the "center of gravity" of its distribution -- the long-run average if you repeated the experiment many times.

$$E[X] = \sum_{i} x_i \cdot P(x_i) \quad \text{(discrete)}$$

**Investment meaning:** The expected return of a stock is the probability-weighted average of all possible returns. It's your best single-number forecast of what the return will be.

**Worked Example:** A stock has these possible returns next year:

| Scenario | Probability | Return |
|---|---|---|
| Boom | 25% | +30% |
| Normal | 50% | +10% |
| Recession | 25% | -15% |

$$E[R] = 0.25 \times 0.30 + 0.50 \times 0.10 + 0.25 \times (-0.15) = 0.075 + 0.05 - 0.0375 = 8.75\%$$

### Variance: Quantifying Risk

Variance measures the spread of possible outcomes around the expected value:

$$\text{Var}(X) = E[(X - \mu)^2] = \sum_i P(x_i)(x_i - \mu)^2$$

The standard deviation $\sigma = \sqrt{\text{Var}(X)}$ is in the same units as the returns, making it easier to interpret.

**Investment meaning:** Variance (or standard deviation) is the most common measure of **total risk** for a single asset. Higher variance means more uncertainty about the actual return.

### Covariance and Correlation: How Assets Move Together

$$\text{Cov}(X, Y) = E[(X - \mu_X)(Y - \mu_Y)]$$

$$\rho_{XY} = \frac{\text{Cov}(X,Y)}{\sigma_X \sigma_Y}, \quad -1 \leq \rho \leq 1$$

**Investment meaning:** Covariance and correlation measure how two assets move *together*. Negative correlation means they tend to move in opposite directions -- exactly what we want for diversification!

### Portfolio Expected Return & Variance (2 assets)

$$E[R_p] = w_1 E[R_1] + w_2 E[R_2]$$
$$\sigma_p^2 = w_1^2 \sigma_1^2 + w_2^2 \sigma_2^2 + 2 w_1 w_2 \text{Cov}(R_1, R_2)$$

> **Key Concept:** Portfolio return is always the weighted average of individual returns. But portfolio *risk* can be **less** than the weighted average of individual risks -- that's the benefit of diversification, and it works because of the covariance term.

> **CFA Exam Tip:** The portfolio variance formula is the most important formula in portfolio theory. Make sure you can compute it quickly. The covariance term is what makes diversification work -- if $\text{Cov} < 0$, it *reduces* portfolio risk.

Let's compute everything for a stock-bond portfolio under three economic scenarios.

In [ ]:
def expected_value(outcomes, probabilities):
    """E[X] = sum(x_i * p_i)"""
    outcomes = np.asarray(outcomes, dtype=float)
    probabilities = np.asarray(probabilities, dtype=float)
    assert np.isclose(np.sum(probabilities), 1.0)
    return np.sum(outcomes * probabilities)


def variance_discrete(outcomes, probabilities):
    """Var(X) = E[X^2] - (E[X])^2"""
    outcomes = np.asarray(outcomes, dtype=float)
    probabilities = np.asarray(probabilities, dtype=float)
    mu = expected_value(outcomes, probabilities)
    return np.sum(probabilities * (outcomes - mu) ** 2)


def covariance_discrete(x_outcomes, y_outcomes, joint_probs):
    """Compute Cov(X,Y) from joint distribution."""
    x = np.asarray(x_outcomes, dtype=float)
    y = np.asarray(y_outcomes, dtype=float)
    jp = np.asarray(joint_probs, dtype=float)
    p_x = np.sum(jp, axis=1)
    p_y = np.sum(jp, axis=0)
    mu_x = np.sum(x * p_x)
    mu_y = np.sum(y * p_y)
    e_xy = 0
    for i in range(len(x)):
        for j in range(len(y)):
            e_xy += x[i] * y[j] * jp[i, j]
    return e_xy - mu_x * mu_y


def portfolio_return_variance(w, mu, cov_matrix):
    """Portfolio expected return and variance."""
    w = np.asarray(w, dtype=float)
    mu = np.asarray(mu, dtype=float)
    cov = np.asarray(cov_matrix, dtype=float)
    port_return = w @ mu
    port_var = w @ cov @ w
    return port_return, port_var


# ── Example: Stock return under different economic scenarios
scenarios = ['Boom', 'Normal', 'Recession']
probs = [0.25, 0.50, 0.25]
stock_returns = [0.30, 0.10, -0.15]
bond_returns = [0.02, 0.06, 0.12]

e_stock = expected_value(stock_returns, probs)
e_bond = expected_value(bond_returns, probs)
var_stock = variance_discrete(stock_returns, probs)
var_bond = variance_discrete(bond_returns, probs)

cov_sb = sum(p * (rs - e_stock) * (rb - e_bond) 
             for p, rs, rb in zip(probs, stock_returns, bond_returns))
corr_sb = cov_sb / (np.sqrt(var_stock) * np.sqrt(var_bond))

print(f"{'Scenario':<12} {'Prob':>6} {'Stock':>8} {'Bond':>8}")
print("-" * 36)
for s, p, rs, rb in zip(scenarios, probs, stock_returns, bond_returns):
    print(f"{s:<12} {p:>6.2f} {rs:>8.2%} {rb:>8.2%}")

print(f"\n{'E[R]':<12} {'':>6} {e_stock:>8.2%} {e_bond:>8.2%}")
print(f"{'σ':<12} {'':>6} {np.sqrt(var_stock):>8.2%} {np.sqrt(var_bond):>8.2%}")
print(f"\nCov(Stock, Bond) = {cov_sb:.6f}")
print(f"Corr(Stock, Bond) = {corr_sb:.4f}")
print(f"\nNegative correlation: stocks and bonds diversify each other!")

# ── 60/40 portfolio
w = np.array([0.60, 0.40])
mu_vec = np.array([e_stock, e_bond])
cov_mat = np.array([[var_stock, cov_sb], [cov_sb, var_bond]])

port_ret, port_var = portfolio_return_variance(w, mu_vec, cov_mat)
print(f"\n60/40 Portfolio:")
print(f"  E[R_p] = {port_ret:.2%}")
print(f"  σ_p    = {np.sqrt(port_var):.2%}")

**Interpreting the results:**

Notice the key numbers:
- **Stocks:** Expected return 8.75% with 15.8% standard deviation (high risk, high reward)
- **Bonds:** Expected return 6.50% with 3.5% standard deviation (lower risk, lower reward)
- **Correlation: -0.99** -- stocks and bonds move in almost perfectly opposite directions in this scenario setup (bonds rally during recessions when stocks fall)

The **60/40 portfolio** delivers 7.85% expected return with just 8.8% standard deviation. Compare this to holding 100% stocks (8.75% return, 15.8% risk). By giving up less than 1% of expected return, you cut risk nearly in half! That's the power of diversification with negatively correlated assets.

Let's visualize the risk-return tradeoff for all possible stock/bond allocations.

In [ ]:
# ── Visualization: Portfolio risk-return for different weights
weights_stock = np.linspace(0, 1, 100)
port_rets = []
port_stds = []

for ws in weights_stock:
    w_vec = np.array([ws, 1 - ws])
    ret, var = portfolio_return_variance(w_vec, mu_vec, cov_mat)
    port_rets.append(ret)
    port_stds.append(np.sqrt(var))

fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(np.array(port_stds) * 100, np.array(port_rets) * 100, 
                c=weights_stock, cmap='coolwarm', s=20)
plt.colorbar(sc, label='Weight in Stocks')

ax.plot(np.sqrt(var_stock) * 100, e_stock * 100, 'o', color=PRIMARY, markersize=12, 
        label='100% Stocks', zorder=5)
ax.plot(np.sqrt(var_bond) * 100, e_bond * 100, 's', color=SECONDARY, markersize=12, 
        label='100% Bonds', zorder=5)
ax.plot(np.sqrt(port_var) * 100, port_ret * 100, '*', color=TERTIARY, markersize=15, 
        label='60/40 Portfolio', zorder=5)

ax.set_xlabel('Risk (Standard Deviation, %)')
ax.set_ylabel('Expected Return (%)')
ax.set_title('Portfolio Efficient Frontier: Stocks vs Bonds')
ax.legend()
plt.tight_layout()
plt.show()

**Reading the efficient frontier:**

The curve shows every possible combination of stocks and bonds. Notice that it bows to the **left** -- this means some portfolios have less risk than *either* individual asset! This is the visual proof of diversification.

The minimum-risk portfolio is somewhere around 20-25% stocks, where the curve reaches its leftmost point. Even adding a small amount of stocks to a bond portfolio can actually *reduce* total risk because of the negative correlation.

---
## 6. Counting Methods

### Why Counting Matters

Counting methods let you figure out how many ways something can happen, which is essential for computing probabilities. In finance, they show up in portfolio construction (how many portfolios can you build from 20 stocks?) and in risk calculations.

### Factorial

$$n! = n \cdot (n-1) \cdot \ldots \cdot 1, \quad 0! = 1$$

### Permutations (order matters)

"How many ways can you *arrange* $k$ items from $n$?"

$$P(n, k) = \frac{n!}{(n-k)!}$$

**Example:** How many ways can you rank the top 3 stocks from 10 candidates? $P(10,3) = 10 \times 9 \times 8 = 720$.

### Combinations (order doesn't matter)

"How many ways can you *choose* $k$ items from $n$?"

$$C(n, k) = \binom{n}{k} = \frac{n!}{k!(n-k)!}$$

**Example:** How many different 5-stock portfolios can you build from 20 candidates? $C(20,5) = 15{,}504$.

> **CFA Exam Tip:** Use permutations when the order of selection matters (ranking analysts, sequencing investments). Use combinations when order doesn't matter (selecting stocks for a portfolio).

Let's implement these from scratch and apply them to finance.

In [ ]:
def factorial(n):
    """Compute n! from scratch."""
    if n <= 1:
        return 1
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result


def permutations(n, k):
    """P(n,k) = n! / (n-k)!"""
    return factorial(n) // factorial(n - k)


def combinations(n, k):
    """C(n,k) = n! / (k! * (n-k)!)"""
    return factorial(n) // (factorial(k) * factorial(n - k))


def multinomial(n, groups):
    """Multinomial coefficient: n! / (n1! * n2! * ... * nk!)"""
    assert sum(groups) == n, "Groups must sum to n"
    denom = 1
    for g in groups:
        denom *= factorial(g)
    return factorial(n) // denom


# ── Portfolio selection: choosing 5 stocks from 20 candidates
n_universe = 20
n_select = 5
n_portfolios = combinations(n_universe, n_select)
print(f"Choosing {n_select} stocks from {n_universe}: C({n_universe},{n_select}) = {n_portfolios:,} possible portfolios")

# Ranking: how many ways to rank top 3 from 10 stocks?
n_rank = permutations(10, 3)
print(f"Ranking top 3 from 10: P(10,3) = {n_rank:,} orderings")

# Multinomial: allocate 12 analysts into 3 teams of 4
n_allocations = multinomial(12, [4, 4, 4])
print(f"Allocating 12 analysts into 3 teams of 4: {n_allocations:,} ways")

# ── Application: Probability of specific portfolio composition
# From 20 stocks (8 tech, 7 healthcare, 5 finance), 
# pick 5 at random. P(exactly 2 tech, 2 healthcare, 1 finance)?
favorable = combinations(8, 2) * combinations(7, 2) * combinations(5, 1)
total = combinations(20, 5)
prob = favorable / total
print(f"\nP(2 tech, 2 healthcare, 1 finance out of 5) = {favorable}/{total} = {prob:.4f}")

**Interpreting the results:**

There are 15,504 different 5-stock portfolios you can build from just 20 stocks. This number grows astronomically with a larger universe -- for the S&P 500 with 500 stocks, there are over $2.6 \times 10^{12}$ possible 5-stock portfolios. This is why systematic portfolio construction methods are essential.

The probability calculation shows that if you randomly pick 5 stocks from a universe of 20 (8 tech, 7 healthcare, 5 finance), there's about a 19% chance of getting exactly 2 tech, 2 healthcare, and 1 finance. This is a hypergeometric probability -- useful for understanding random sampling from stratified populations.

---
## References

1. **CFA Institute**, *CFA Program Curriculum Level I*, Quantitative Methods: Probability Concepts.
2. **DeFusco, R., McLeavey, D., Pinto, J., & Runkle, D.**, *Quantitative Investment Analysis*, 3rd ed., CFA Institute/Wiley, 2015.
3. **Ross, S.**, *A First Course in Probability*, 10th ed., Pearson, 2019.
4. **Bertsekas, D. & Tsitsiklis, J.**, *Introduction to Probability*, 2nd ed., Athena Scientific, 2008.